# Task 2: Count Occurrences of Each Name

## Step 1: Initialize Spark & Load Valid Data
Reuse parsing logic from Task 1.

In [4]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

# Load and clean data (same as Task 1)
raw_rdd = sc.textFile("/home/jovyan/data/raw/employees.txt")
header = raw_rdd.first()

data_rdd = raw_rdd.filter(lambda line: line != header) \
                  .filter(lambda line: line.strip() != "") \
                  .map(lambda line: line.split(","))

# Validation
EXPECTED_COLS = 9

def validate(record):
    if len(record) != EXPECTED_COLS:
        return False
    try:
        float(record[4])
        return True
    except ValueError:
        return False

valid_rdd = data_rdd.filter(validate)

print(f"Valid records loaded: {valid_rdd.count()}")

Valid records loaded: 9


## Step 2: Extract Names & Count Occurrences

Map each record to `(name, 1)` then reduce by key.

In [5]:
# Extract name (index 1) and map to (name, 1)
name_pairs = valid_rdd.map(lambda rec: (rec[1], 1))

# Reduce by key: sum counts per name
name_counts = name_pairs.reduceByKey(lambda a, b: a + b)

print("Name counts (sorted):")
for name, count in name_counts.collect():
    print(f"  {name:<20} → {count}")

Name counts (sorted):
  Sarah Johnson        → 1
  Michael Williams     → 1
  Jennifer Brown       → 1
  David Jones          → 1
  Lisa Garcia          → 1
  Patricia Wilson      → 1
  James Anderson       → 1
  Mary Thomas          → 1
  John Smith           → 1


## Step 3: Sort by Count Descending

Most frequent names first.

In [6]:
# Sort by count descending
sorted_counts = name_counts.map(lambda x: (x[1], x[0])) \
                           .sortByKey(ascending=False) \
                           .map(lambda x: (x[1], x[0]))

print("Names sorted by frequency:")
for name, count in sorted_counts.collect():
    bar = "█" * count
    print(f"  {name:<20} {bar} ({count})")


Names sorted by frequency:
  Sarah Johnson        █ (1)
  Michael Williams     █ (1)
  Jennifer Brown       █ (1)
  David Jones          █ (1)
  Lisa Garcia          █ (1)
  Patricia Wilson      █ (1)
  James Anderson       █ (1)
  Mary Thomas          █ (1)
  John Smith           █ (1)


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 38860)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

##  Task 2 Complete

All names are unique → each appears exactly once.